# ㅇㅇ

In [5]:
pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 6.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from scipy import stats

%matplotlib inline
from matplotlib import font_manager, rc

font_location = "C:\Windows\Fonts\malgun.ttf"
font_name = font_manager.FontProperties(fname=font_location).get_name()
rc('font', family=font_name)
plt.rcParams['axes.unicode_minus'] = False


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import mutual_info_classif
from lightgbm import LGBMClassifier
import random

# -------------------------
# 0) 데이터 로드 & 라벨 생성
# -------------------------
df = pd.read_csv("data/2024년도.CSV", encoding="cp949")
df["고혈당_분석용"] = (df["식전혈당(공복혈당)"] >= 100).astype(int)

# 🔹 데이터 샘플링 (빠른 테스트용: 10만행만 사용)
df_work = df.sample(n=100000, random_state=42).copy()

# -------------------------
# 1) 기본 세팅
# -------------------------
TARGET = "고혈당_분석용"
ID_LIKE = ["기준년도", "가입자일련번호"]
GLUCOSE_COLS = ["식전혈당(공복혈당)"]
MISSING_THRESH = 0.30
LOW_REL_TOP_K = 15
N_TRIALS = 10          # 🔹 빠른 테스트용: 10회만
K_RANGE = (3, 5)

ANCHORS = ["연령대코드(5세단위)", "허리둘레", "수축기혈압", "이완기혈압",
           "HDL콜레스테롤", "LDL콜레스테롤", "흡연상태", "음주여부"]
ANCHORS = [c for c in ANCHORS if c in df_work.columns]

# -------------------------
# 2) Feature Pool 설정
# -------------------------
exclude_cols = set([TARGET] + ID_LIKE + [c for c in GLUCOSE_COLS if c in df_work.columns])
feature_pool = [c for c in df_work.columns if c not in exclude_cols]

# 결측치 30% 넘는 컬럼 제외
missing_rate = df_work[feature_pool].isna().mean().sort_values()
kept_by_missing = missing_rate[missing_rate <= MISSING_THRESH].index.tolist()

# -------------------------
# 3) 랜덤 조합 평가 함수
# -------------------------
def eval_random_combos(df, pool_cols, y_col, n_trials=10, k_range=(3,5),
                       always_include=None, random_state=42, verbose_every=5):
    rng = random.Random(random_state)
    results = []
    pool_cols = [c for c in pool_cols if c not in (always_include or [])]
    y = df[y_col].astype(int)

    for i in range(1, n_trials+1):
        k = rng.randint(k_range[0], k_range[1])
        subset = rng.sample(pool_cols, min(k, len(pool_cols)))
        if always_include:
            subset = list(set(subset + list(always_include)))

        X = df[subset].copy()
        for c in X.columns:
            if X[c].nunique() <= 15:
                X[c] = X[c].fillna(X[c].mode().iloc[0])
            else:
                X[c] = X[c].astype(float).fillna(X[c].median())

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=i
        )

        neg, pos = (y_train==0).sum(), (y_train==1).sum()
        scale = max(1.0, neg / max(1, pos))

        model = LGBMClassifier(
            n_estimators=100,     # 🔹 트리 개수 줄임 (400 → 100)
            learning_rate=0.1,    # 🔹 조금 높여서 수렴 빠르게
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=i,
            class_weight={0:1, 1:scale},
            n_jobs=-1
        )
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:,1]
        auc = roc_auc_score(y_test, proba)

        results.append({
            "trial": i,
            "k": len(subset),
            "cols": subset,
            "AUC": round(auc, 4)
        })

        if verbose_every and i % verbose_every == 0:
            print(f"… {i}/{n_trials} done (last AUC={auc:.3f})")

    return pd.DataFrame(results).sort_values(by="AUC", ascending=False)

# -------------------------
# 4) 실행 예시
# -------------------------
res_test = eval_random_combos(
    df=df_work,
    pool_cols=kept_by_missing,
    y_col=TARGET,
    n_trials=N_TRIALS,
    k_range=K_RANGE,
    always_include=ANCHORS,
    random_state=123
)

print("\n=== 빠른 테스트 결과 TOP 5 ===")
print(res_test.head(5))


[LightGBM] [Info] Number of positive: 31686, number of negative: 48314
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003541 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1087
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number of positive: 31729, number of negative: 48271
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002046 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 865
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[Li

In [11]:
# =========================
# 0) 샘플링 (10만건)
# =========================
df_sample = df_work.sample(n=100000, random_state=42)

# =========================
# 1) 치과 컬럼 포함 랜덤 조합
# =========================
DENTAL_EXTRA = [c for c in ["치아우식증유무", "치석"] if c in df_sample.columns]

res_with_dental = eval_random_combos(
    df=df_sample,
    pool_cols=low_rel_candidates,    
    y_col=TARGET,
    n_trials=20,                     # 빠른 확인 → 20번만
    k_range=(5, 8),                  # 조합 크기 5~8
    always_include=ANCHORS + DENTAL_EXTRA,   
    random_state=123
)

print("\n=== 치과 변수 포함 (샘플 10만건, TOP 결과 10) ===")
print(res_with_dental.head(10))


NameError: name 'low_rel_candidates' is not defined

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from lightgbm import LGBMClassifier
import shap

# -----------------------------
# 데이터 준비
# -----------------------------
target = "고혈당_분석용"
exclude_cols = ["고혈당_분석용", "고혈당_서비스용", "식전혈당(공복혈당)", "risk_level"]
X = df.drop(columns=exclude_cols, errors="ignore")
y = df[target].astype(int)

cat_cols = ["성별코드", "흡연상태", "음주여부"]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# 결측치 처리 + 스케일링
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_imputed = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imputed)

# -----------------------------
# LightGBM (풀 피처)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgb_model.fit(X_train, y_train)

y_pred_proba = lgb_model.predict_proba(X_test)[:, 1]
print("=== LightGBM (풀 피처) ===")
print("AUC:", roc_auc_score(y_test, y_pred_proba))

# -----------------------------
# SHAP 기반 중요도 Top 10
# -----------------------------
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_test[:1000])  # 샘플 일부만 (속도 문제 방지)

shap_importance = np.abs(shap_values[1]).mean(axis=0)  # 클래스=1 기준
shap_df = pd.DataFrame({"feature": X.columns, "shap_importance": shap_importance})
shap_top10 = shap_df.sort_values("shap_importance", ascending=False).head(10)
print("\n=== SHAP Top 10 피처 ===")
print(shap_top10)

# -----------------------------
# RFE 기반 중요 피처
# -----------------------------
log_model = LogisticRegression(max_iter=500, solver="liblinear")
rfe = RFE(log_model, n_features_to_select=10)
rfe.fit(X_scaled, y)

rfe_support = pd.DataFrame({
    "feature": X.columns,
    "selected": rfe.support_,
    "ranking": rfe.ranking_
}).sort_values("ranking")

print("\n=== RFE Top 10 피처 ===")
print(rfe_support[rfe_support["selected"]])

# -----------------------------
# Top 피처만으로 다시 모델 학습
# -----------------------------
top_features = shap_top10["feature"].tolist()
X_train_top, X_test_top = X_train[top_features], X_test[top_features]

lgb_model_top = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgb_model_top.fit(X_train_top, y_train)

y_pred_proba_top = lgb_model_top.predict_proba(X_test_top)[:, 1]
print("\n=== LightGBM (SHAP Top 10 피처만) ===")
print("AUC:", roc_auc_score(y_test, y_pred_proba_top))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
import shap

# -----------------------------
# 1. 피처 / 라벨 분리
# -----------------------------
target = "고혈당_분석용"
exclude_cols = ["고혈당_분석용", "식전혈당(공복혈당)", "고혈당_서비스용", "risk_level"]
X = df.drop(columns=exclude_cols, errors="ignore")
y = df[target].astype(int)

# -----------------------------
# 2. 결측치 처리 (null 제거)
# -----------------------------
# 범주형 후보
cat_cols = ["성별코드", "흡연상태", "음주여부"]
num_cols = [c for c in X.columns if c not in cat_cols]

# 수치형 → 중앙값
for col in num_cols:
    if col in X.columns:
        X[col] = X[col].fillna(X[col].median())

# 범주형 → 최빈값
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].fillna(X[col].mode()[0])

# 원-핫 인코딩
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# 스케일링 (옵션: 로지스틱 대비용, LGBM은 안 해도 OK)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# 3. 모델 학습
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

lgb = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgb.fit(X_train, y_train)

# -----------------------------
# 4. SHAP 계산
# -----------------------------
explainer = shap.TreeExplainer(lgb)
shap_values = explainer.shap_values(X_test)

# 중요도 평균
shap_importance = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": X.columns,
    "shap_importance": shap_importance
}).sort_values(by="shap_importance", ascending=False)

print("=== SHAP Top 10 피처 ===")
print(importance_df.head(10))


In [ ]:
# -----------------------------
# Null만 있는 컬럼 제거 (중요!)
# -----------------------------
X = X.dropna(axis=1, how="all")   # 전부 NaN인 컬럼 삭제

# 결측치 처리 + 스케일링
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_imputed = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imputed)

# 다시 DataFrame으로 (RFE/SHAP에서 컬럼명 보존)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE
from lightgbm import LGBMClassifier
import shap

# -----------------------------
# 0. 데이터 준비
# -----------------------------
# (유빈님 CSV 불러오기)
df = pd.read_csv("국민건강보험공단_건강검진정보_2024.CSV", encoding="cp949")

# 라벨 생성 (100 이상 → 1)
df["고혈당_분석용"] = (df["식전혈당(공복혈당)"] >= 100).astype(int)

target = "고혈당_분석용"
y = df[target]

# 제외할 컬럼
exclude_cols = ["고혈당_분석용", "고혈당_서비스용", "식전혈당(공복혈당)", "risk_level"]
X = df.drop(columns=exclude_cols, errors="ignore")

# -----------------------------
# 1. 전부 NaN 컬럼 제거
# -----------------------------
null_only_cols = [c for c in X.columns if X[c].isna().all()]
print("⚠️ 전부 NaN인 컬럼:", null_only_cols)

X = X.drop(columns=null_only_cols)

# -----------------------------
# 2. 범주형 원-핫 인코딩
# -----------------------------
cat_cols = ["성별코드", "흡연상태", "음주여부"]
X = pd.get_dummies(X, columns=[c for c in cat_cols if c in X.columns], drop_first=True)

# -----------------------------
# 3. 결측치 처리 + 스케일링
# -----------------------------
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_imputed = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imputed)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
y = y.astype(int)

# -----------------------------
# 4. RFE (Logistic Regression 기반)
# -----------------------------
log_model = LogisticRegression(max_iter=500, solver="liblinear")
rfe = RFE(log_model, n_features_to_select=10)
rfe.fit(X_scaled_df, y)

rfe_support = pd.DataFrame({
    "feature": X.columns,
    "selected": rfe.support_,
    "ranking": rfe.ranking_
}).sort_values("ranking")

print("\n=== RFE Top 10 피처 ===")
print(rfe_support[rfe_support["selected"]])

# -----------------------------
# 5. LightGBM + SHAP
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42
)

lgb_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(X_train, y_train)

# SHAP 값 계산
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_test)

shap_importance = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    "feature": X.columns,
    "shap_importance": shap_importance
}).sort_values("shap_importance", ascending=False)

print("\n=== SHAP Top 10 피처 ===")
print(shap_df.head(10))

# -----------------------------
# 6. 성능 확인
# -----------------------------
y_pred = lgb_model.predict(X_test)
y_proba = lgb_model.predict_proba(X_test)[:,1]

print("\n=== LightGBM 성능 ===")
print(f"ACC = {accuracy_score(y_test, y_pred):.3f}")
print(f"F1  = {f1_score(y_test, y_pred):.3f}")
print(f"AUC = {roc_auc_score(y_test, y_proba):.3f}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# 이미 y_test, y_pred_proba (확률) 가 있다고 가정
thresholds = [0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7]

results = []
for t in thresholds:
    y_pred = (y_pred_proba >= t).astype(int)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append({
        "Threshold": t,
        "Precision": round(prec, 3),
        "Recall": round(rec, 3),
        "F1": round(f1, 3)
    })

df_thresh = pd.DataFrame(results)
print(df_thresh)


In [ ]:
# -----------------------------
# 1. BMI 추가
# -----------------------------
df["BMI"] = df["체중(5kg단위)"] * 5 / ((df["신장(5cm단위)"]*5 / 100) ** 2)

# -----------------------------
# 2. 피처/타깃 분리
# -----------------------------
target = "고혈당_분석용"
y = df[target].astype(int)

exclude_cols = ["고혈당_분석용", "고혈당_서비스용", "식전혈당(공복혈당)", "risk_level"]
X = df.drop(columns=exclude_cols, errors="ignore")

cat_cols = ["성별코드", "흡연상태", "음주여부"]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X = X.astype("float32").fillna(0)

# -----------------------------
# 3. 모델 학습
# -----------------------------
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale = neg / pos if pos > 0 else 1

model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0: 1, 1: scale},
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 예측 확률
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 성능 확인
y_pred = (y_pred_proba >= 0.5).astype(int)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("=== BMI 추가 후 모델 성능 ===")
print(f"Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}, AUC: {auc:.3f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from lightgbm import LGBMClassifier

# -----------------------------
# 1. 혈압차 추가
# -----------------------------
df["혈압차"] = df["수축기혈압"] - df["이완기혈압"]

# -----------------------------
# 2. 피처/타깃 분리
# -----------------------------
target = "고혈당_분석용"
y = df[target].astype(int)

exclude_cols = ["고혈당_분석용", "고혈당_서비스용", "식전혈당(공복혈당)", "risk_level"]
X = df.drop(columns=exclude_cols, errors="ignore")

cat_cols = ["성별코드", "흡연상태", "음주여부"]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X = X.astype("float32").fillna(0)

# -----------------------------
# 3. train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale = neg / pos if pos > 0 else 1

# -----------------------------
# 4. LightGBM 학습
# -----------------------------
model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0: 1, 1: scale},
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# -----------------------------
# 5. 성능 평가
# -----------------------------
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("=== 혈압차 추가 후 모델 성능 ===")
print(f"Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}, AUC: {auc:.3f}")


In [ ]:
df["콜레스테롤비율"] = df["총콜레스테롤"] / (df["HDL콜레스테롤"] + 1e-6)
df["AST_ALT비율"] = df["혈청지오티(AST)"] / (df["혈청지피티(ALT)"] + 1e-6)

# -----------------------------
# 2. 피처/타깃 분리
# -----------------------------
target = "고혈당_분석용"
y = df[target].astype(int)

exclude_cols = ["고혈당_분석용","고혈당_서비스용","식전혈당(공복혈당)","risk_level"]
X = df.drop(columns=exclude_cols, errors="ignore")

cat_cols = ["성별코드","흡연상태","음주여부"]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X = X.astype("float32").fillna(0)

# -----------------------------
# 3. 모델 학습 & Threshold 실험
# -----------------------------
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

neg, pos = (y_train==0).sum(), (y_train==1).sum()
scale = neg / pos if pos > 0 else 1

model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0:1, 1:scale},
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 예측 확률
y_pred_proba = model.predict_proba(X_test)[:,1]

# 여러 Threshold에서 성능 확인
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
for t in thresholds:
    y_pred = (y_pred_proba >= t).astype(int)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    print(f"Threshold={t:.2f} → Precision={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}, AUC={auc:.3f}")

In [ ]:
# -----------------------------
# 1. 새로운 파생변수 생성
# -----------------------------
df["BMI"] = df["체중(5kg단위)"] * 5 / ((df["신장(5cm단위)"]*5 / 100) ** 2)   # 단위 보정
df["혈압차"] = df["수축기혈압"] - df["이완기혈압"]
df["콜레스테롤비율"] = df["총콜레스테롤"] / (df["HDL콜레스테롤"] + 1e-6)
df["AST_ALT비율"] = df["혈청지오티(AST)"] / (df["혈청지피티(ALT)"] + 1e-6)

# -----------------------------
# 2. 피처/타깃 분리
# -----------------------------
target = "고혈당_분석용"
y = df[target].astype(int)

exclude_cols = ["고혈당_분석용","고혈당_서비스용","식전혈당(공복혈당)","risk_level"]
X = df.drop(columns=exclude_cols, errors="ignore")

cat_cols = ["성별코드","흡연상태","음주여부"]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X = X.astype("float32").fillna(0)

# -----------------------------
# 3. 모델 학습 & Threshold 실험
# -----------------------------
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

neg, pos = (y_train==0).sum(), (y_train==1).sum()
scale = neg / pos if pos > 0 else 1

model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0:1, 1:scale},
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 예측 확률
y_pred_proba = model.predict_proba(X_test)[:,1]

# 여러 Threshold에서 성능 확인
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
for t in thresholds:
    y_pred = (y_pred_proba >= t).astype(int)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    print(f"Threshold={t:.2f} → Precision={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}, AUC={auc:.3f}")


In [8]:
import pickle
import numpy as np

# -----------------------------
# 1. 모델 학습 (이미 하신 부분)
# -----------------------------
model.fit(X_train, y_train)

# -----------------------------
# 2. 모델 & 메타데이터 저장
# -----------------------------
meta = {
    "features": X.columns.tolist(),   # 피처 순서 저장
    "median_values": X.median().to_dict()  # 중앙값 저장
}

with open("glucose_model.pkl", "wb") as f:
    pickle.dump((model, meta), f)


NameError: name 'model' is not defined